# 7 Working at the vector level

Every notebook so far has fed **raw data** to a model: pixels in, logits out. This one does
something different, and it is the pattern the rest of the course is built on.

We split the problem in two:

1. **Encoding** — turn each item into a vector, *once*. Expensive, needs the hardware, and the
   result is written to disk as an artefact.
2. **Heading** — train a small network *on top of those vectors*. Cheap, done hundreds of times,
   runs fine on a laptop.

The interesting consequence is that step 1 is a **choice**, and different choices give you
different vectors for the same data. We will make that choice twice on the same 25.000 movie
reviews:

- **designed features** — 43 numbers you can read and explain, written by hand with regexes;
- **a learned embedding** — 384 numbers from a pretrained language model, which nobody can read.

Same task, same head, same trainer. Only the representation changes. At the end we compare what
each one costs and what each one buys — which is the question this course keeps coming back to.

In [ ]:
from pathlib import Path
import warnings

import torch
from datasets import load_dataset

warnings.simplefilter("ignore", UserWarning)

ARTEFACTS = Path("../../artefacts").resolve()
ARTEFACTS.mkdir(exist_ok=True)

## 6.1 The data

IMDB: 25.000 train and 25.000 test movie reviews, labelled positive or negative. Straight from the
hub, nothing to download by hand.

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
train = dataset["train"]
test = dataset["test"]
train

In [ ]:
print(train[0]["label"], "|", train[0]["text"][:300])

## 6.2 Representation 1: features you design yourself

A `RegexVectorizer` counts hand-written patterns: it is as simple as searching for key-words like "good", "bad", "great", etc. Each pattern that survives becomes one binary
feature:

```python
def build_imdb_review_pattern() -> re.Pattern:
    """Match film-related vocabulary in IMDB reviews.

    Each match is a single word/phrase, e.g. "great", "terrible", "acting",
    "horror". After fitting, each unique matched term becomes one binary
    feature dimension (1 if the word appears anywhere in the text, 0 if not).

    Examples:
        "The acting was great but the plot was terrible"
        → matches: ["acting", "great", "plot", "terrible"]
    """
    sentiment = (
        "great|good|bad|poor|excellent|terrible|awful|brilliant|boring|"
        "amazing|weak|superb|dreadful|outstanding|mediocre|impressive|"
        "masterpiece|hilarious|touching|predictable|formulaic"
    )
    genre = (
        "horror|comedy|thriller|drama|romance|action|western|"
        "documentary|musical|mystery|fantasy|animation"
    )
    craft = (
        "acting|performance|screenplay|script|direction|plot|"
        "cinematography|dialogue|soundtrack|editing"
    )
    return re.compile(rf"\b(?:{sentiment}|{genre}|{craft})\b", re.IGNORECASE)
```

It is fitted on the training texts — patterns that occur in fewer than `min_doc_frequency`
documents are dropped, because a feature that fires twice in 25.000 reviews cannot teach a model
anything.

The `RegexVectorizer` will turn a text into a binary vector: `[0, 0, 1, 0, 1, ...]` where every `1` means "yes we found this word in the text.

In [ ]:
from vectormesh import RegexVectorizer
from vectormesh.data.vectorizers import build_imdb_review_pattern, harmonize_imdb_match

regexvectorizer = RegexVectorizer(
    pattern_builder=build_imdb_review_pattern,
    harmonizer=harmonize_imdb_match,
    min_doc_frequency=15,
    max_features=200,
    device="cpu",
    training_texts=train["text"],  # fit it on the full train set
)
regexvectorizer.get_metadata

In [ ]:
regexvectorizer.print_stats()

43 patterns survived. **This is the whole appeal of a designed representation: you can read it.**
Every one of those 43 numbers has a name and a reason, and if the model learns that feature 12
matters, you can go look up what feature 12 is.

In [ ]:
batch = train.shuffle(seed=42)[:4]
out = regexvectorizer(batch["text"])

print(f"the shape of the regex features for 4 samples is {torch.stack(out['regex_features']).shape}")
print(f"a single binary vector looks like this: {out['regex_features'][0]}")

`(4, 43)`: four reviews, 43 features each. One review on its own is a `(43,)` **vector** — the
first rung of the ladder ([tensor-ladder.md](tensor-ladder.md)).

Formally that is a rank-1 tensor and nothing more. Semantically it is "one item, one
representation", which is what the rest of this notebook will keep being about.

## 6.3 The cache: computing it once

Running the vectorizer over 25.000 reviews on every epoch would be pure waste — the vectors do not
change, the encoder is frozen. So we run it once and write the result to disk as a **`VectorCache`**.

For regexes this is almost free (a couple of seconds). Do it anyway, because the exact same call
will shortly be handling something that takes two hours.

In [ ]:
from vectormesh import VectorCache

traincache = VectorCache.create(
    cache_dir=ARTEFACTS,
    vectorizer=regexvectorizer,
    dataset=train,
    dataset_tag="imdb_train",
)
testcache = VectorCache.create(
    cache_dir=ARTEFACTS,
    vectorizer=regexvectorizer,
    dataset=test,
    dataset_tag="imdb_test",
)

### A cache is a contract

The cache is not just a folder of numbers — it ships a `metadata.json` that tells you what is in
it. **That file is the public API of a dataset**, and reading it is the first thing you should do
with any cache, especially the ones you did not build yourself.

In [ ]:
traincache.metadata

Everything you need in order to write the model code is in there:

- `hidden_size: 43` — the `D` of every shape downstream, so the head's input size;
- `tensordtype: 1` — one item is a rank-1 `(D)` tensor, so batching is a plain `torch.stack` and
  **no padding or aggregation is needed**;
- `num_observations: 25000` — how many rows;
- `features` — which columns exist.

Read that before you write a line of model code and you already know the shapes. We will come back
to this idea in §6.7 with a cache where the answer is more interesting.

## 6.4 From cache to batches

Two small pieces stand between the cache and the model.

**`OneHot`** turns the integer label into a `(2,)` target vector. **`Collate`** is the `collate_fn`
you wrote by hand in `03_dataloader.ipynb`: it takes a list of rows and produces a batch. It needs
to know which column holds the vectors, which holds the target, and how to stack them.

In [ ]:
from vectormesh.data import Collate, OneHot

onehot = OneHot(num_classes=2, label_col="label", target_col="onehot")
train_oh = traincache.map(onehot)
test_oh = testcache.map(onehot)

In [ ]:
from torch.utils.data import DataLoader

collate_fn = Collate(
    embedding_col="regex_features",
    target_col="onehot",
    padder=torch.stack,
)

trainloader = DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(test_oh, batch_size=32, shuffle=False, collate_fn=collate_fn)

X, y = next(iter(trainloader))
X.shape, y.shape

`(43,)` per item became `(32, 43)` per batch: **collation is the step that adds the batch axis**.
On the ladder we moved from `(D)` to `(B, D)`.

That is the shape a `Linear` layer wants, so there is nothing else to do — we can go straight to a
head.

## 6.5 The head

`NeuralNet` is two `Linear` layers with a GELU in between. `Serial` chains components, the way
`nn.Sequential` does. With one component it looks like overkill; in §6.8 the same `Serial` will
hold three.

In [ ]:
from vectormesh.components import NeuralNet, Serial

hidden_size = traincache.metadata["regex_features"]["hidden_size"]
print(f"hidden size should be the number of regex features: {hidden_size}")
pipeline = Serial([NeuralNet(hidden_size=hidden_size, out_size=2)])
pipeline(X).shape

`(B, D) -> (B, K)`: 32 items in, 32 predictions over 2 classes out. The last rung.

In [ ]:
import torch.optim as optim
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components.metrics import F1Score
from vectormesh.data.vectorizers import detect_device

device = detect_device()
loss_fn = torch.nn.CrossEntropyLoss()
log_dir = Path("demo").absolute()

settings = TrainerSettings(
    epochs=2, # increase to see better results
    metrics=[F1Score()],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TENSORBOARD],
    earlystop_kwargs={"save": False, "verbose": True, "patience": 3},
)
print(f"device: {device}, {len(trainloader)} train steps and {len(testloader)} valid steps per epoch")

In [ ]:
import time

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

start = time.time()
trainer.loop()
regex_traintime = time.time() - start
print(f"trained in {regex_traintime:.0f}s")

Note down that F1. 43 hand-written binary features, and the whole thing — fitting the vectorizer,
building the cache, and five epochs of training — took under a minute.



## 6.6 Representation 2: a learned embedding

`Vectorizer` wraps a pretrained transformer. We use
`ibm-granite/granite-embedding-small-english-r2`, a small English embedding model: text in, a
384-dimensional vector out. Nobody designed those 384 numbers — they fell out of pretraining on a
large corpus, and no single one of them means anything you can name.

There is one wrinkle. The model has a **context window** of 512 tokens, and plenty of IMDB reviews
are longer than that. `Vectorizer` handles it by **chunking**: a long review becomes several
overlapping 512-token windows, each embedded separately.

So one review is no longer a `(D)` vector. It is a `(C, D)` matrix, with `C` varying per review.
That is a different rung, and it will change what we have to build.

In [ ]:
from vectormesh import Vectorizer

model_name = "ibm-granite/granite-embedding-small-english-r2"
vectorizer = Vectorizer(model_name=model_name, col_name="embed", max_length=512)
vectorizer.get_metadata


In [ ]:
embeddings = vectorizer(inputs=["this is a test", "this is another test"], batchsize=2)

print(f"this turned the first sentence into a tensor of shape {embeddings['embed'][0].shape}")

### Getting the cache

Here the split from the top of the notebook stops being theoretical. Embedding 25.000 reviews took
**16 minutes on an M-series GPU and about two hours on a CPU** — per split. That is not something
to do in a lesson, and it is not something *anyone* should do twice.

So it was done once, with the vectorizer you just built:

```python
traincache = VectorCache.create(
    cache_dir=ARTEFACTS,
    vectorizer=vectorizer,      # the only line that differs from §6.3
    dataset=train,
    dataset_tag="imdb_train",
)
```

Look at that against the regex cache in §6.3 and notice what did *not* change. Same call, same
arguments, same three lines — one of them takes two seconds and the other takes two hours. The
cost lives in the encoder, not in your code, and that is exactly what makes the artefact worth
publishing: whoever pays the two hours pays it for everybody.

The result is on the hub, so all you do is download it. Nothing below rebuilds anything.


In [ ]:
from huggingface_hub import snapshot_download

CACHE_REPO = "pttrn-io/imdb-granite-embedding-small-english-r2"


path = Path(snapshot_download(
    CACHE_REPO,
    repo_type="dataset",
))

traincache = VectorCache.load(path / "train")
testcache = VectorCache.load(path / "test")

## 6.7 Read the metadata before writing any model code

We now have a cache someone else may well have built. Before touching a model, find out what is in
it.

In [ ]:
traincache.metadata

Three things to read off it, in order:

- **`hidden_size: 384`** — so `D = 384`, and the head's input size is 384 rather than 43.
- **`tensordtype: 2`** — one item is a rank-2 `(C, D)` tensor. This is the important one: items no
  longer have the same shape as each other, so `torch.stack` will refuse to batch them. We need a
  **padder**. And a batch will be `(B, C, D)`, which is one rung too high for a `Linear` head, so
  we also need an **aggregator** to get back down to `(B, D)`.
- **`chunk_sizes`** — the histogram of `C`. This tells us how much padding will cost.

That is the whole architecture, derived from a JSON file, before importing anything.

In [ ]:
import matplotlib.pyplot as plt

chunks = {int(k): v for k, v in traincache.metadata["embed"]["chunk_sizes"].items()}
total = sum(chunks.values())
for c in sorted(chunks):
    print(f"{c} chunk(s): {chunks[c]:6d}  ({chunks[c] / total:5.1%})")

plt.bar(list(chunks.keys()), list(chunks.values()), color="skyblue", edgecolor="black")
plt.xlabel("chunks per review (C)")
plt.ylabel("number of reviews")
plt.title("How long are the reviews, in chunks?")

A long tail, but a thin one: the overwhelming majority of reviews fit in one or two chunks.

That makes `max_chunks` a genuine, decidable trade-off rather than a number to guess. Set it to 2
and you truncate a few hundred long reviews. Set it to 8 and you keep everything, but every batch
is then mostly zeros — you would be spending four times the compute to serve less than 1% of the
data. We take 2.

In [ ]:
from vectormesh.components import FixedPadding, MaskedMeanAggregator

C = 2
collate_fn = Collate(
    embedding_col="embed",
    target_col="onehot",
    padder=FixedPadding(max_chunks=C),
)

trainloader = DataLoader(traincache, batch_size=32, shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(testcache, batch_size=32, shuffle=False, collate_fn=collate_fn)

X, y = next(iter(trainloader))
X.shape, y.shape

`(32, 2, 384)` — `(B, C, D)`, one rung higher than the regex path's `(32, 43)`.

Hand that straight to a `Linear(384, 2)` and, as you saw in `01_tensors.ipynb` §1.8, **it will not
complain**: it acts on the last axis and would give you `(32, 2, 2)`, a prediction per chunk. That
is the silent-failure case, and it is why we aggregate first.

In [ ]:
aggregator = MaskedMeanAggregator()
aggregator(X).shape

`(B, C, D) -> (B, D)`. **Aggregation is the only rank-reducing step in the whole pipeline**, which
is why it is the most consequential structural choice you make.

Why *masked* mean? Because most of those `C=2` slots are zero padding. A plain mean averages the
zeros in, dragging every short review's vector toward the origin — and 86% of our reviews are short.
`MaskedMeanAggregator` skips the all-zero rows. (Exercise E1 below asks you to measure exactly how
much that matters.)

In [ ]:
hidden_size = traincache.metadata["embed"]["hidden_size"]

pipeline = Serial([
    MaskedMeanAggregator(),                              # (B, C, D) -> (B, D)
    NeuralNet(hidden_size=hidden_size, out_size=2),      # (B, D)    -> (B, K)
])
pipeline(X).shape

In [ ]:
settings = TrainerSettings(
    epochs=2, # increase to see better results
    metrics=[F1Score()],
    logdir=log_dir,
    train_steps=len(trainloader),
    valid_steps=len(testloader),
    reporttypes=[ReportTypes.TENSORBOARD],
    earlystop_kwargs={"save": False, "verbose": True, "patience": 3},
)

trainer = Trainer(
    model=pipeline,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainloader,
    validdataloader=testloader,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau,
    device=device,
)

start = time.time()
trainer.loop()
embed_traintime = time.time() - start
print(f"trained in {embed_traintime:.0f}s")

## 6.8 The comparison

Same 25.000 reviews, same task, same head, same trainer. Only the representation changed.

In [ ]:
def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


regex_head = Serial([NeuralNet(hidden_size=43, out_size=2)])

print(f"{'':22} {'regex':>12} {'granite':>12}")
print(f"{'dimensions D':22} {43:>12} {384:>12}")
print(f"{'head parameters':22} {count_params(regex_head):>12,} {count_params(pipeline):>12,}")
print(f"{'training (5 epochs)':22} {regex_traintime:>11.0f}s {embed_traintime:>11.0f}s")
print(f"{'one-time encoding':22} {'~3s':>12}    {'~16min(GPU)/~2h(CPU)':>12}")

Read that table carefully, because it is the shape of nearly every decision in this course.

**Both training runs are cheap.** On my machine the regex head took 11 seconds and the embedding
head 44 seconds — the embedding path is four times slower, because each batch carries 384
dimensions and two chunks through an aggregator instead of 43 numbers straight into a `Linear`.
Four times slower, and still under a minute. Both of them are things you can run twenty times in
an afternoon while you try ideas out, which is the entire point.

**The encoding costs differ by a factor of a thousand.** Three seconds against two hours. And it
is paid *once*, by whoever has the hardware, and then it is an artefact everybody reuses. If you
had to re-encode every epoch, the embedding path would cost hours per experiment and this notebook
would not exist.

**What you get for it**, on this task, is an F1 of about 0.79 against 0.73. Real, but smaller than
the thousandfold difference in encoding cost might have led you to expect — and worth sitting with,
because 43 patterns a person wrote in an afternoon got 92% of the way to a pretrained transformer.

Also look at the losses in that second run: test loss bottoms out around epoch 2 and then climbs
while train loss keeps falling. 148.610 parameters on 25.000 examples is enough to start memorising.
The regex head, with 1.980 parameters, never gets the chance. Small heads on frozen vectors are
hard to overfit, which is a genuine advantage of this setup and not just an economy.

> Those validation numbers are only trustworthy because of a fix that landed alongside this
> refactor. `mltrainer` used to call `next(iter(loader))` inside its step loop, which rebuilds the
> iterator every step — so a `shuffle=False` validation loader kept handing back its *first batch*,
> `valid_steps` times over. Every F1 in the old version of this notebook described 32 reviews.
> Requires `mltrainer >= 0.3.2`.

Neither representation is the right answer. Be precise about the trade:

| | designed (regex) | learned (embedding) |
|---|---|---|
| readable | yes — every feature has a name | no |
| debuggable | yes — look up feature 12 | only by probing |
| encoding cost | seconds | hours, once |
| needs fitting on your data | yes | no, it is pretrained |
| transfers to a new task | no, the patterns are IMDB-specific | yes, the same vectors |
| catches what you did not anticipate | no | often |
| parameters to overfit with | few | many |

A regex feature set is unbeatable when you have to justify a decision to a regulator, and it is the
cheapest thing that can possibly work. A pretrained embedding is unbeatable when the signal lives in
phrasing you would never have thought to write a pattern for.

**Knowing which one you are reaching for, and why, is the skill.** The rest of this course is mostly
variations on this question: which parts of your pipeline are designed, which are learned, and what
happens when you move the line.

## Exercises

**E1 — the aggregator ablation.** Swap `MaskedMeanAggregator` for `MeanAggregator` (a plain mean
over the chunk axis, padding included) and retrain. *Before you run it*, predict which one wins and
by roughly how much, using the `chunk_sizes` histogram from §6.7. Then run it, and explain the
result in terms of what zero-padding does to a mean.

This is your first ablation, and the point is that you could have reasoned it out. Not every
experiment needs running to be understood — but running it is how you find out whether your
reasoning was right.

**E2 — how much data does each representation need?** Shrink the training set to 200 rows
(`traincache.select(range(200))`) and re-run both representations. Does the 43-dimensional
hand-written vector close the gap on the 384-dimensional embedding? Where would you expect the
crossover to be, and why?

**E3 — read a cache you did not build.** Find a `metadata.json` in `artefacts/` for a cache you did
not create in this notebook. Without running any code, state:

1. the shape of one item;
2. the shape of a batch of 32, and which padder (if any) gets you there;
3. every component you would need between the `DataLoader` and the loss.

Then check yourself by building the loader.